# Simulation Sanity Check

Load a saved simulation pickle, validate its numerical and process-level outputs, and inspect standard figures.

## Execution paths

1. **Inspection (default):** load the selected pickle and run every following section without changing it.
2. **Optional re-run:** set `RERUN_SIMULATION = True`, define any parameter overrides in the dedicated section, and re-run the loaded configuration before validation.

Run the notebook from the campaign directory so pickle discovery starts at the correct location. The selected pickle should contain detailed trajectories; performance-mode results can lack data required by some plots.

**Workflow:**
1. Setup and configuration
2. Load or optionally re-run the simulation
3. KPI, feasibility, and elemental-closure diagnostics
4. Batch, kinetic, and process-level visualizations
5. Interpretation checklist

## 1. Setup and Configuration

In [67]:
from pathlib import Path

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.simulation_data import SimulationData

print("✓ Imports successful")

✓ Imports successful


### 1.1 Discover Available Simulation Pickles

Scan the campaign directory recursively for saved simulation pickles. Configure the selected result and inspection scope in the next cell.

In [68]:
# ── Scan this campaign directory recursively for simulation pickles ───────────
CAMPAIGN_DIR = Path.cwd()
PICKLE_FILES = sorted(
    path for path in CAMPAIGN_DIR.rglob("*.pkl")
    if path.is_file()
)

if not PICKLE_FILES:
    raise FileNotFoundError(
        f"No .pkl files found below the campaign directory: {CAMPAIGN_DIR.resolve()}"
    )

print(f"Found {len(PICKLE_FILES)} simulation pickle(s) below {CAMPAIGN_DIR.resolve()}:\n")
for number, path in enumerate(PICKLE_FILES, start=1):
    print(f"[{number:>3}] {path.relative_to(CAMPAIGN_DIR)}")


Found 19 simulation pickle(s) below C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird:

[  1] 01_FedBatch\export\fedbatch_constant_p2_20260818_1233\opt_20260818_1233.pkl
[  2] 01_FedBatch\export\fedbatch_constant_p2_20260824_2230\opt_20260824_2230.pkl
[  3] 01_FedBatch\export\fedbatch_csfr_p2_20260818_1246\opt_20260818_1246.pkl
[  4] 01_FedBatch\export\fedbatch_exponential_p2_20260818_1301\opt_20260818_1301.pkl
[  5] 01_FedBatch\export\fedbatch_gsp_p2_20260818_1458\opt_20260818_1458.pkl
[  6] 01_FedBatch\export\fedbatch_linear_p2_20260818_1514\opt_20260818_1514.pkl
[  7] 02_Batch\export\batch_p2_20260818_1521\opt_20260818_1521.pkl
[  8] 02_Batch\export\batch_p2_20260824_2236\opt_20260824_2236.pkl
[  9] 03_Chemostat\export\chemostat_p2_20260818_1531\opt_20260818_1531.pkl
[ 10] 04_ConcentratedFedBatch\export\cf_constant_p2_20260818_1654\opt_20260818_1654.pkl
[ 11] 04_ConcentratedFedBatch\export\cf_cspr_p2_20260818_1846\

In [69]:
# ── Select the saved simulation and inspection scope ──────────────────────────
# Set this to one of the numbers printed by the preceding discovery cell.
PICKLE_NUMBER = 2

# Individual batch used by the batch and kinetic figure sections.
BATCH_NUMBER = 1

# None includes every batch in process-level overlays; use an integer to limit them.
MAX_BATCHES = None

# False preserves the loaded result. Set True only to apply overrides and re-run.
RERUN_SIMULATION = True

if not isinstance(PICKLE_NUMBER, int) or isinstance(PICKLE_NUMBER, bool):
    raise TypeError("PICKLE_NUMBER must be an integer.")
if not 1 <= PICKLE_NUMBER <= len(PICKLE_FILES):
    raise ValueError(
        f"PICKLE_NUMBER must be between 1 and {len(PICKLE_FILES)}; "
        f"received {PICKLE_NUMBER}."
    )
if not isinstance(BATCH_NUMBER, int) or isinstance(BATCH_NUMBER, bool) or BATCH_NUMBER < 1:
    raise ValueError("BATCH_NUMBER must be a positive integer.")
if MAX_BATCHES is not None and (
    not isinstance(MAX_BATCHES, int) or isinstance(MAX_BATCHES, bool) or MAX_BATCHES < 1
):
    raise ValueError("MAX_BATCHES must be None or a positive integer.")

PICKLE_PATH = PICKLE_FILES[PICKLE_NUMBER - 1]

print(f"Selected pickle [{PICKLE_NUMBER}]: {PICKLE_PATH.relative_to(CAMPAIGN_DIR)}")
print(f"Batch for detailed plots: {BATCH_NUMBER}")
print(f"Optional re-run enabled: {RERUN_SIMULATION}")

Selected pickle [2]: 01_FedBatch\export\fedbatch_constant_p2_20260824_2230\opt_20260824_2230.pkl
Batch for detailed plots: 1
Optional re-run enabled: True


## 2. Load the Selected Simulation

This is the default inspection path. The optional re-run section below replaces `sim` and `data` only when enabled.

In [70]:
sim = CellCultureSimulation.load_pickle(PICKLE_PATH, base_export_dir=PICKLE_PATH.parent)
data = sim.data

if not 1 <= BATCH_NUMBER <= len(data.batches):
    raise ValueError(
        f"BATCH_NUMBER must be between 1 and {len(data.batches)} for the selected simulation; "
        f"received {BATCH_NUMBER}."
    )

print(f"✓ Loaded: {data.simulation_type}")
print(f"  Source           : {PICKLE_PATH.resolve()}")
print(f"  Batches          : {len(data.batches)}")
print(f"  Simulation time  : {data.simulated_window_totals_stats.get('total_simulation_time_h', 'N/A'):.1f} h")
print(f"  Parameters name  : {data.parameters.metadata.get('name', 'N/A') if data.parameters else 'N/A'}")

CellCultureSimulation loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\01_FedBatch\export\fedbatch_constant_p2_20260824_2230\opt_20260824_2230.pkl
✓ Loaded: Fedbatch
  Source           : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\01_FedBatch\export\fedbatch_constant_p2_20260824_2230\opt_20260824_2230.pkl
  Batches          : 5
  Simulation time  : 960.0 h
  Parameters name  : base_parameters


### 2.1 Optional Parameter Overrides and Re-run

Leave `RERUN_SIMULATION = False` to inspect the loaded result unchanged. To explore a changed configuration, enable it in the configuration cell and add parameter-path/value pairs to `PARAMETER_OVERRIDES` in the next cell. The re-run replaces `sim` and `data` for every subsequent diagnostic and figure.

### 2.2 Run Optional Overrides

In [71]:
# Keep this mapping empty to re-run the saved configuration without changes.
# Uncomment or add entries only when RERUN_SIMULATION is True.
PARAMETER_OVERRIDES = {
    # "fed_batch.C_Gln_feed": 10,
    # "fed_batch.C_AA_feed": 10,
    # "initial_conditions.C_Glc_init": 40,
    # "initial_conditions.C_Gln_init": 20,
    # "fed_batch.CSFR": 5e-12,
    # "perfusion.CSPR": 10.0e-11,
}

if RERUN_SIMULATION:
    for parameter_path, value in PARAMETER_OVERRIDES.items():
        sim.parameters[parameter_path] = value

    sim = CellCultureSimulation(
        parameters=sim.parameters,
        kinetics=sim.kinetics,
        gas_controller=sim.gas_controller,
        operating_mode=sim.operating_mode,
        feed_strategy=sim.feed_strategy,
        perfusion_strategy=sim.perfusion_strategy,
        stoichiometry_mode=sim.stoichiometry_mode,
        performance_mode=sim.performance_mode,
        solver_method="LSODA",
    )
    data = sim.run_simulation()
    print(f"✓ Re-ran simulation with {len(PARAMETER_OVERRIDES)} override(s).")
else:
    print("✓ Inspection mode: using the loaded simulation unchanged.")

run_simulation execution time: 508.370 ms
✓ Re-ran simulation with 0 override(s).


## 3. Validation and Diagnostics

Review optimization feasibility, performance, and elemental closure before interpreting figures.

### 3.1 Optimization Feasibility Check

This uses the same `ObjectiveFunction` path and rejection criteria as differential evolution. A pass means the result is feasible for optimization; it does not replace scientific review.

In [72]:
# Evaluate the current simulation with the same ObjectiveFunction implementation
# used by differential evolution. This avoids maintaining a second, drifting copy
# of the optimization rejection logic in the notebook.

from animakinetica.analysis.optimization import ObjectiveFunction
import numpy as np

WALL_TIMEOUT_S = 10.0  # Match the campaign optimization timeout when diagnosing runtime rejections.

# ObjectiveFunction needs objectives and weights even though this cell only
# evaluates feasibility. Use a harmless available numeric KPI with zero weight.
validation_function = ObjectiveFunction(
    baseline_simulation=sim,
    opt_config={},
    objectives=["STY_kg_m3_day"],
    weights=[0.0],
    timeout=WALL_TIMEOUT_S,
    baseline_values=None,
    debug_rejections=True,
)

# An empty optimization configuration maps this empty vector to the current
# baseline parameters, then runs the same temporary-simulation path and every
# ObjectiveFunction rejection check. Valid evaluations return a non-positive
# score; soft constraint penalties and hard rejections return positive scores.
filter_score = float(validation_function(np.array([], dtype=float)))
passes_filter = filter_score <= 0.0

print("ObjectiveFunction filter check:")
print(f"  {'PASS' if passes_filter else 'FAIL'}  valid ObjectiveFunction score <= 0")
print(f"  Score: {filter_score:.6g}")

# Supplementary diagnostics only; all pass/fail decisions remain in
# ObjectiveFunction. The fields below expose inputs to its current filters.
simulated_totals = data.simulated_window_totals_stats
expected_window_h = validation_function._get_expected_simulation_window_target_h()
soft_violations = validation_function._get_soft_constraint_violations(
    data,
    simulated_totals,
)

try:
    closure = data.compute_boundary_closure()
    elem_res = {
        element: abs(values["relative_residual_pct"])
        for element, values in closure["elements"].items()
    }
except (ValueError, KeyError):
    elem_res = None

total_base_in_L = float(simulated_totals.get("total_base_in_L", 0.0) or 0.0)
total_acid_in_L = float(simulated_totals.get("total_acid_in_L", 0.0) or 0.0)
total_medium_in_L = float(simulated_totals.get("total_medium_in_L", 0.0) or 0.0)
acid_base_fraction = (
    (total_base_in_L + total_acid_in_L) / total_medium_in_L
    if total_medium_in_L > 0.0 else None
)

print("\nDiagnostics:")
print(f"  total_simulation_time_h = {float(simulated_totals.get('total_simulation_time_h', 0.0) or 0.0):.3f}")
print(f"  expected_simulation_window_h = {expected_window_h:.3f}")
print(f"  total_active_cultivation_time_h = {float(simulated_totals.get('total_active_cultivation_time_h', 0.0) or 0.0):.3f}")
print(f"  projection_window_h = {simulated_totals.get('projection_window_h')}")
print(f"  projection_target_time_h = {simulated_totals.get('projection_target_time_h')}")
print(f"  number_of_batches = {simulated_totals.get('number_of_batches')}")
print(f"  total_DCM_kg = {simulated_totals.get('total_DCM_kg', data.summary_stats.get('total_DCM_kg'))}")
print(f"  last_batch_termination_reason = {simulated_totals.get('last_batch_termination_reason')}")
print(f"  unique_batch_termination_reasons = {simulated_totals.get('unique_batch_termination_reasons')}")
print(f"  batch_final_volume_min_fraction = {simulated_totals.get('batch_final_volume_min_fraction')}")
print(f"  batch_final_volume_max_fraction = {simulated_totals.get('batch_final_volume_max_fraction')}")
print(f"  last_batch_final_volume_fraction = {simulated_totals.get('last_batch_final_volume_fraction')}")
print(f"  total_base_in_L = {total_base_in_L:.6g}")
print(f"  total_acid_in_L = {total_acid_in_L:.6g}")
print(f"  total_medium_in_L = {total_medium_in_L:.6g}")
print(f"  acid_base_fraction = {acid_base_fraction if acid_base_fraction is not None else 'N/A'}")
print(f"  phase_active_runtime_h = {simulated_totals.get('phase_active_runtime_h')}")
print(f"  batch_phase_active_runtime_h = {simulated_totals.get('batch_phase_active_runtime_h')}")
print(f"  ObjectiveFunction soft_constraint_violations = {soft_violations}")
if elem_res is not None:
    print(
        "  boundary closure |relative_residual_pct| "
        f"(soft limit {ObjectiveFunction._MAX_CLOSURE_PCT:.0f}%) = {elem_res}"
    )

run_simulation execution time: 471.515 ms
ObjectiveFunction filter check:
  PASS  valid ObjectiveFunction score <= 0
  Score: -0

Diagnostics:
  total_simulation_time_h = 960.000
  expected_simulation_window_h = 960.000
  total_active_cultivation_time_h = 0.000
  projection_window_h = 960.0
  projection_target_time_h = 7680.0
  number_of_batches = 5
  total_DCM_kg = None
  last_batch_termination_reason = Maximum time reached
  unique_batch_termination_reasons = ['Max volume', 'Maximum time reached']
  batch_final_volume_min_fraction = 0.9884900356441534
  batch_final_volume_max_fraction = 1.00000005
  last_batch_final_volume_fraction = 0.9884900356441534
  total_base_in_L = 194.04
  total_acid_in_L = 0
  total_medium_in_L = 7900.37
  acid_base_fraction = 0.024560870355609447
  phase_active_runtime_h = {'BATCH': 340.89796986289974, 'FEDBATCH': 619.1020301371004}
  batch_phase_active_runtime_h = [{'batch_number': 1, 'BATCH': 116.56543897184936, 'FEDBATCH': 122.8842266642427}, {'batch_num

### 3.2 Performance Summary

Inspect the primary KPIs and cultivation-level totals.

In [73]:
print("Performance Metrics:")
for key, value in data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key:<45} {value:.4f}")
    else:
        print(f"  {key:<45} {value}")

Performance Metrics:
  total_DCM_harvested_kg                        146.2159
  total_WCM_harvested_kg                        487.3865
  STY_kg_m3_day                                 0.2285
  STY_WCM_kg_m3_day                             0.7615
  medium_conversion_efficiency_m3_kg            0.4323
  medium_conversion_efficiency_m3_kg_WCM        0.1297
  protein_conversion_efficiency_kg_kg           3.8577
  feed_conversion_efficiency_kg_kg              8.3071
  feed_conversion_efficiency_kg_kg_WCM          2.4921


In [74]:
# Cultivation summary table
data.get_cultivation_summary_df()

,Batch No.,Termination Reason,Cultivation Time (h),Batch Start Time (h),Batch End Time (h),Accum. Cultivation Time (d),Final Glucose (mmol/L),Final Glutamine (mmol/L),Final Lactate (mmol/L),Final Ammonia (mmol/L),...,Accum. Acid Addition Use (L),Accum. Total Inlet Use (L),Harvest Volume (L),Accum. Harvest Volume (L),Final Cell Density (10^6 cells/mL),Final Cell Density (g/L),Cells Harvested (10^12 Cells),Cells Harvested (kg),Accum. Cell Harvest (10^12 Cells),Accum. Biomass Harvest (kg)
0,1,Max volume,239.449666,0.000000,239.449666,9.977069,38.736073,8.027514,50.669093,7.442918,...,0.0,1999.996505,1569.120473,1569.120473,2.785001,2.506501,4.370001,3.933001,4.370001,3.933001
1,2,Max volume,180.599486,239.449666,420.049152,17.502048,36.006092,2.793041,49.088459,7.210734,...,0.0,3569.113991,1531.710370,3100.830843,2.562516,2.306265,3.925033,3.532529,8.295034,7.465531
2,3,Max volume,180.158641,420.049152,600.207793,25.008658,35.647101,1.191962,47.914378,7.038270,...,0.0,5100.821681,1514.652548,4615.483391,2.472455,2.225210,3.744910,3.370419,12.039944,10.83595
3,4,Max volume,180.611540,600.207793,780.819333,32.534139,35.995792,0.638146,46.822925,6.877944,...,0.0,6615.471467,1501.964182,6117.447573,2.409465,2.168518,3.618930,3.257037,15.658874,14.092987
4,5,Maximum time reached,179.180667,780.819333,960.000000,40.000000,36.593119,0.422566,45.727791,6.717077,...,0.0,8094.412627,1976.980071,8094.427644,2.351513,2.116362,4.648895,4.184005,20.307769,18.276992


In [75]:
process_stoich_closure = data.compute_stoichiometric_closure()
process_stoich_closure_df = data.build_closure_summary_df(process_stoich_closure)

process_stoich_biomass = next(
    spec for spec in process_stoich_closure['component_specs']
    if spec['label'] == 'Dry biomass'
)

print('Integrated stoichiometric CHON closure:')
display(process_stoich_closure_df)
print(f"Max absolute elemental residual: {process_stoich_closure['max_abs_residual']:.6g} mol-atoms")

Integrated stoichiometric CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,2338.144228,2338.144218,-0.000010,-4.198578e-07
1,H,4680.823122,4680.823100,-0.000022,-4.718844e-07
2,O,2555.419561,2555.419554,-0.000007,-2.881167e-07
3,N,224.810197,224.810195,-0.000002,-1.091680e-06


Max absolute elemental residual: 2.20881e-05 mol-atoms


In [76]:
process_external_elemental_closure = data.compute_boundary_closure()
process_external_elemental_closure_df = data.build_closure_summary_df(process_external_elemental_closure)

process_dry_biomass = next(
    spec for spec in process_external_elemental_closure['component_specs']
    if spec['label'] == 'DCM out'
 )

print('Whole-process external-boundary CHON closure:')
display(process_external_elemental_closure_df)
print(f"Max absolute elemental residual: {process_external_elemental_closure['max_abs_residual']:.6g} mol-atoms")

Whole-process external-boundary CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,5539.755321,5539.751203,-0.004119,-7.435036e-05
1,H,909247.584081,909249.229527,1.645446,1.809678e-04
2,O,454677.402193,454678.231696,0.829502,1.824376e-04
3,N,572.826102,572.826100,-0.000002,-3.788246e-07


Max absolute elemental residual: 1.64545 mol-atoms


## 4. Batch-State and Operating-Control Figures

Use `BATCH_NUMBER` from the configuration section to inspect biomass, metabolites, volume/feed, gas transfer, and pH control for one batch.

## 5. Batch Plots

In [77]:
# Cell density & metabolites
fig = data.plot_biomass_metabolites(batch_number=BATCH_NUMBER)
fig.show()

In [78]:
# Volume & feed / dilution rate
fig = data.plot_volume_feed(batch_number=BATCH_NUMBER)
fig.show()

In [79]:
# Gas transfer (DO, agitation, gas flow, O2 fraction)
fig = data.plot_gas_transfer(batch_number=BATCH_NUMBER)
fig.show()

In [80]:
# pH plots
fig = data.plot_pH_control(batch_number=BATCH_NUMBER)
fig.show()

## 5. Kinetic-Behavior Figures

Review nutrient limitation, inhibition, kinetic rates, and stoichiometric yields for the selected batch.

In [81]:
# Nutrient limitation factors (Monod terms)
fig = data.plot_nutrient_factors(batch_number=BATCH_NUMBER)
fig.show()

In [82]:
# Inhibition factors (lactate, ammonia, O2, osmolality)
fig = data.plot_inhibition_factors(batch_number=BATCH_NUMBER)
fig.show()

In [83]:
# Specific rates (mu, q_Glc, q_Gln, q_Lac, q_O2, etc.)
fig = data.plot_kinetic_rates(batch_number=BATCH_NUMBER)
fig.show()

In [84]:
# Stoichiometric yield coefficients (Y_X/Glc, Y_Lac/Glc, Y_O2/Glc, etc.)
fig = data.plot_stoichiometric_yields(batch_number=BATCH_NUMBER)
fig.show()

## 6. Multi-Batch Process Figures

Use `MAX_BATCHES` from the configuration section to constrain batch overlays and cumulative totals.

In [85]:
# Overlaid state variable profiles across batches
figs = data.plot_profiles(max_batches=MAX_BATCHES)
for fig in figs:
    fig.show()

In [86]:
# Cumulative harvest (biomass & volume) across batches
fig = data.plot_cumulative_harvest(max_batches=MAX_BATCHES)
fig.show()

In [87]:
# Cumulative medium use
fig = data.plot_cumulative_medium_use(max_batches=MAX_BATCHES)
fig.show()

In [88]:
# Cumulative gas use
fig = data.plot_cumulative_gas_use(max_batches=MAX_BATCHES)
fig.show()

## 7. Interpretation Checklist

Before accepting a result:

- Confirm the selected pickle and whether the optional re-run path was enabled.
- Investigate a failed optimization feasibility check, especially its rejection diagnostics and termination reason.
- Review elemental residuals in both closure reports; large residuals require mass-balance investigation.
- Check the selected-batch figures for unexpected depletion, volume limits, gas-control saturation, or pH-control behavior.
- If a plot lacks data, verify that the pickle was generated with `performance_mode=False` and contains detailed trajectories.